# Fashion Retrieval — Colab: öznitelik çıkarma (Adım 2)

**Ne yapar:** Drive'daki `crops.zip`'i Colab diskine açar, GitHub'daki kodu çeker ve dört hazır omurgayla
(ResNet-50, CLIP, SigLIP, DINOv2) `val_internal` ve `test` kırpıntılarının özniteliklerini çıkarır.
Çıktılar Drive'a (`CMPE9103_Retrieval/features/`) yazılır; oturum kapansa da kaybolmaz.

**Nasıl kullanılır:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **GPU** seçin (T4 yeterli, L4/A100 daha hızlı).
Sonra hücreleri sırayla çalıştırın (Shift+Enter). Yarıda kalırsa baştan çalıştırın; bitmiş listeler atlanır.

In [ ]:
!nvidia-smi -L

## 1. Ayarlar

In [ ]:
GITHUB_USER = "ethembelka"
REPO_NAME   = "fashion-product-retrieval"
DRIVE_ROOT  = "/content/drive/MyDrive/CMPE9103_Retrieval"
CROPS_ZIP   = f"{DRIVE_ROOT}/data/crops.zip"
FEATURES    = f"{DRIVE_ROOT}/features"        # çıktılar buraya
PROCESSED   = "/content/processed"             # crops/ ve splits/ burada açılır (hızlı yerel disk)

MODELS = ["resnet50", "clip_vitb16", "siglip_vitb16", "dinov2_vitb14"]
SPLITS = ["val_internal", "test"]

## 2. Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(FEATURES, exist_ok=True)
assert os.path.exists(CROPS_ZIP), f"crops.zip bulunamadı: {CROPS_ZIP}"
print("crops.zip:", round(os.path.getsize(CROPS_ZIP)/1e9, 2), "GB")

## 3. Kodu GitHub'dan çek

Depo private olduğu için bir GitHub token gerekir: Colab'da sol menü **🔑 Secrets** → `GITHUB_TOKEN` adıyla ekleyin (Not defteri erişimi açık).

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
if os.path.exists(f"/content/{REPO_NAME}"):
    !git -C /content/{REPO_NAME} pull -q
else:
    !git clone -q {url} /content/{REPO_NAME}
%cd /content/{REPO_NAME}
!git log --oneline -1
!pip install -q timm open_clip_torch faiss-cpu pyarrow

## 4. crops.zip'i Colab diskine aç (ilk seferde ~5–10 dk)

In [ ]:
import time
if not os.path.exists(f"{PROCESSED}/crops"):
    t = time.time()
    !cp "{CROPS_ZIP}" /content/crops.zip
    !mkdir -p {PROCESSED} && unzip -q /content/crops.zip -d {PROCESSED} && rm /content/crops.zip
    print(f"açıldı, {time.time()-t:.0f} s")
!ls {PROCESSED} && ls {PROCESSED}/splits && du -sh {PROCESSED}/crops

## 5. Öznitelikleri çıkar

Her model × ayrım için `gallery`, `queries`, `openset` listeleri işlenir. Var olan çıktılar atlanır.

In [ ]:
for m in MODELS:
    for s in SPLITS:
        print(f"\n===== {m} / {s} =====")
        !python scripts/extract_features.py --model {m} --split {s} --processed {PROCESSED} --features {FEATURES} --workers 4

## 6. Kontrol

Aşağıda her model/ayrım için dosya sayıları ve hız özetleri. Hepsi tamamsa Claude Code'a 'öznitelikler hazır' deyin; değerlendirme yerelde (CPU) yapılacak.

In [ ]:
import json, glob
for rep in sorted(glob.glob(f"{FEATURES}/*/*/extract_report.json")):
    r = json.load(open(rep))
    done = {k: v.get("n") for k, v in r["lists"].items()}
    speed = [v.get("images_per_s") for v in r["lists"].values() if "images_per_s" in v]
    print(f"{r['model']:14s} {r['split']:13s} dim={r['embed_dim']:4d} {done} img/s={speed}")